In [ ]:
from cadabra2 import *

# 12d Riemann tensor from the F-theory metric ansatz

We compute the components of the 12d Riemann tensor $\hat{R}_{ABCD}$ from
the block-diagonal metric $\hat{\mathcal{G}}_{MN} = g_{mn} \oplus \mathcal{M}_{ij}$,
where $\mathcal{M}_{ij} = \frac{1}{\tau_2}\begin{pmatrix}|\tau|^2 & \tau_1 \\ \tau_1 & 1\end{pmatrix}$
and all fields depend only on the 10d coordinates $x^m$.

**Key result** (proved below): $\hat{R}_{imjn} = -\tfrac{1}{2}\nabla_n \partial_m \mathcal{M}_{ij}$
(the quadratic Christoffel corrections cancel by $\Gamma^p_{mn}$ symmetry).

**Strategy**: define $\partial_m \mathcal{M}_{ij}$ as cadabra expressions,
then let cadabra compute everything from them.

In [ ]:
{m,n,p,q,r,s,t,u}::Indices(name=tenD, position=fixed).

\partial{#}::PartialDerivative.
\nabla{#}::Derivative.

\tau1::Depends(\partial{#}).
\tau2::Depends(\partial{#}).
P_{m}::Depends(\partial{#}).
\bar{P}_{m}::Depends(\partial{#}).
Q_{m}::Depends(\partial{#}).

DP_{m n}::Symmetric.
DPbar_{m n}::Symmetric.

g_{m n}::Metric.
g^{m n}::InverseMetric.
\Gamma^{p}_{m n}::TableauSymmetry(shape={2}, indices={1,2}).

In [ ]:
# Substitution rules: tau derivatives <-> P, Q
# From P_m = i/(2 tau2) d_m tau and Q_m = -d_m(tau1)/(2 tau2):
#   d_m tau1 = -2 tau2 Q_m
#   d_m tau2 = -tau2 (P_m + Pbar_m)

subs_dt := { \partial_{m}{\tau1} -> -2 \tau2 Q_{m},
             \partial_{m}{\tau2} -> -\tau2 (P_{m} + \bar{P}_{m}) };

subs_dt_n := { \partial_{n}{\tau1} -> -2 \tau2 Q_{n},
               \partial_{n}{\tau2} -> -\tau2 (P_{n} + \bar{P}_{n}) };

# Covariant derivatives of P, Pbar, Q
# (from D_m P_n = nabla_m P_n - 2i Q_m P_n, symmetric)
subs_2nd := { \nabla_{n}{Q_{m}}       -> (1/2) \imath (DP_{n m} - DPbar_{n m})
                                        - Q_{n} (P_{m} + \bar{P}_{m}),
              \nabla_{n}{P_{m}}       -> DP_{n m} + 2 \imath Q_{n} P_{m},
              \nabla_{n}{\bar{P}_{m}} -> DPbar_{n m} - 2 \imath Q_{n} \bar{P}_{m} };

## Step 1: $\partial_m \mathcal{M}_{ij}$ in terms of $P$, $\bar{P}$, $Q$

Elementary calculus + substitution of the $P$, $Q$ rules. The only
simplification done by hand is cancelling $\tau_2 \cdot \tau_2^{-1}$
(cadabra treats $\tau_2$ as an opaque symbol and cannot do this).

$$\partial_m \mathcal{M}_{22} = \partial_m(\tau_2^{-1}) = \frac{P_m + \bar{P}_m}{\tau_2}$$

$$\partial_m \mathcal{M}_{12} = \partial_m(\tau_1/\tau_2) = -2Q_m + \frac{\tau_1}{\tau_2}(P_m+\bar{P}_m)$$

$$\partial_m \mathcal{M}_{11} = \partial_m(|\tau|^2/\tau_2) = -4\tau_1 Q_m + \frac{\tau_1^2-\tau_2^2}{\tau_2}(P_m+\bar{P}_m)$$

In [ ]:
# d_m M_{ij} with free index m
dM22_m := \tau2^{-1} (P_{m} + \bar{P}_{m});
dM12_m := -2 Q_{m} + \tau1 \tau2^{-1} (P_{m} + \bar{P}_{m});
dM11_m := -4 \tau1 Q_{m} + (\tau1^{2} - \tau2^{2}) \tau2^{-1} (P_{m} + \bar{P}_{m});

# Copies with free index n (for hatGamma and second derivatives)
dM22_n := \tau2^{-1} (P_{n} + \bar{P}_{n});
dM12_n := -2 Q_{n} + \tau1 \tau2^{-1} (P_{n} + \bar{P}_{n});
dM11_n := -4 \tau1 Q_{n} + (\tau1^{2} - \tau2^{2}) \tau2^{-1} (P_{n} + \bar{P}_{n});

print("d_m M_22 =", str(dM22_m))
print("d_m M_12 =", str(dM12_m))
print("d_m M_11 =", str(dM11_m))

## Step 2: $\hat{R}_{imjn} = -\tfrac{1}{2} \nabla_n \partial_m \mathcal{M}_{ij}$

For the block-diagonal ansatz with $\partial_i = 0$, the full 12d Riemann is:
$$\hat{R}_{imjn} = -\tfrac{1}{2}\partial_n\partial_m \mathcal{M}_{ij}
+ g_{pq}\hat{\Gamma}^p_{in}\hat{\Gamma}^q_{mj} - g_{pq}\hat{\Gamma}^p_{ij}\hat{\Gamma}^q_{mn}$$
The quadratic term is $-\tfrac{1}{2}\Gamma^q_{mn}\partial_q\mathcal{M}_{ij}$.
Rewriting $\partial_n\partial_m \mathcal{M}_{ij}
= \nabla_n(\partial_m\mathcal{M}_{ij}) + \Gamma^p_{nm}\partial_p\mathcal{M}_{ij}$,
the $\Gamma \cdot \partial\mathcal{M}$ pieces cancel.

Cadabra applies Leibniz + `subs_2nd` to compute $\nabla_n(\partial_m\mathcal{M}_{ij})$.
The Leibniz expansion is written out (tau cancellations noted), then cadabra
handles the covariant derivative substitutions.

In [ ]:
# --- R_{2m2n} = -(1/2) nabla_n(dM22_m) ---
# dM22_m = tau2^{-1} S_m.  Leibniz:
#   nabla_n(tau2^{-1}) * S_m + tau2^{-1} * nabla_n(S_m)
#   nabla_n(tau2^{-1}) = tau2^{-1} S_n  (after tau cancel)

R2m2n := -(1/2) \tau2^{-1} (\nabla_{n}{P_{m}} + \nabla_{n}{\bar{P}_{m}})
        -(1/2) \tau2^{-1} (P_{n} + \bar{P}_{n}) (P_{m} + \bar{P}_{m});
substitute(_, subs_2nd);
collect_terms(_);
print("R_{2m2n} =", str(R2m2n))

In [ ]:
# --- R_{1m2n} = -(1/2) nabla_n(dM12_m) ---
# dM12_m = -2 Q_m + tau1 tau2^{-1} S_m.  Leibniz on each piece:
#   nabla_n(-2 Q_m) -> uses subs_2nd directly
#   nabla_n(tau1 * tau2^{-1} * S_m): 3 factors, Leibniz gives 3 terms
#     nabla_n(tau1) = -2 tau2 Q_n  ->  -2 Q_n S_m  (after tau cancel)
#     nabla_n(tau2^{-1}) = tau2^{-1} S_n
#     nabla_n(S_m) -> uses subs_2nd

R1m2n := \nabla_{n}{Q_{m}}
       + Q_{n} (P_{m} + \bar{P}_{m})
       - (1/2) \tau1 \tau2^{-1} (P_{n} + \bar{P}_{n}) (P_{m} + \bar{P}_{m})
       - (1/2) \tau1 \tau2^{-1} (\nabla_{n}{P_{m}} + \nabla_{n}{\bar{P}_{m}});
substitute(_, subs_2nd);
collect_terms(_);
print("R_{1m2n} =", str(R1m2n))

In [ ]:
# --- R_{1m1n} = -(1/2) nabla_n(dM11_m) ---
# dM11_m = -4 tau1 Q_m + (tau1^2-tau2^2) tau2^{-1} S_m
# Leibniz on -4 tau1 Q_m: 2 terms
#   -4 nabla_n(tau1) Q_m = 8 tau2 Q_n Q_m  ->  after -(1/2): -4 tau2 Q_n Q_m
#   -4 tau1 nabla_n(Q_m)  ->  after -(1/2): 2 tau1 nabla_n Q_m
# Leibniz on (tau1^2-tau2^2) tau2^{-1} S_m: 3 terms
#   nabla_n(tau1^2-tau2^2) = -4 tau1 tau2 Q_n + 2 tau2^2 S_n
#     after * tau2^{-1} and tau cancel: -4 tau1 Q_n + 2 tau2 S_n
#     combined with S_m and -(1/2): 2 tau1 Q_n S_m - tau2 S_n S_m ... wait
#     actually this combines with the (tau1^2+tau2^2)/tau2 S_n S_m term below.
#   nabla_n(tau2^{-1}) * (tau1^2-tau2^2) S_m = (tau1^2-tau2^2) tau2^{-1} S_n S_m
#   nabla_n(S_m) * (tau1^2-tau2^2) tau2^{-1}

R1m1n := -4 \tau2 Q_{n} Q_{m}
       + 2 \tau1 \nabla_{n}{Q_{m}}
       + 2 \tau1 Q_{n} (P_{m} + \bar{P}_{m})
       - (1/2) (\tau1^{2} + \tau2^{2}) \tau2^{-1} (P_{n} + \bar{P}_{n}) (P_{m} + \bar{P}_{m})
       - (1/2) (\tau1^{2} - \tau2^{2}) \tau2^{-1} (\nabla_{n}{P_{m}} + \nabla_{n}{\bar{P}_{m}});
substitute(_, subs_2nd);
collect_terms(_);
print("R_{1m1n} =", str(R1m1n))

## Step 3: $\hat{R}_{mn12}$ (spacetime-spacetime, fiber-fiber)

All $\partial_i = 0$, so only quadratic Christoffel terms survive:
$$\hat{R}_{mnij} = \mathcal{M}_{kl}\bigl(\Omega_m{}^k{}_j\,\Omega_n{}^l{}_i - \Omega_m{}^k{}_i\,\Omega_n{}^l{}_j\bigr)$$

where $\Omega_m{}^i{}_j = \tfrac{1}{2}\mathcal{M}^{ik}\partial_m\mathcal{M}_{kj}$.
The $\Omega$ components are derived from $\mathcal{M}^{-1}\cdot\partial_m\mathcal{M}$
with $\tau_2\cdot\tau_2^{-1}$ cancelled. Cadabra then builds the full quadratic product
and simplifies.

In [ ]:
# Omega_m^i_j = (1/2) M^{ik} d_m M_{kj}
# M^{-1} = [[1/tau2, -tau1/tau2], [-tau1/tau2, |tau|^2/tau2]]
# e.g. Om^1_2 = (1/2)(M^{11} dM_{12} + M^{12} dM_{22})
#             = (1/2)((-2Q/t2 + t1 S/t2^2) + (-t1/t2)(S/t2))
#             = -Q/t2  (after cancelling t1 S/t2^2 terms)

Om11_m := -\tau1 \tau2^{-1} Q_{m} - (1/2)(P_{m} + \bar{P}_{m});
Om12_m := -\tau2^{-1} Q_{m};
Om21_m := (\tau1^{2} - \tau2^{2}) \tau2^{-1} Q_{m} + \tau1 (P_{m} + \bar{P}_{m});
Om22_m :=  \tau1 \tau2^{-1} Q_{m} + (1/2)(P_{m} + \bar{P}_{m});

Om11_n := -\tau1 \tau2^{-1} Q_{n} - (1/2)(P_{n} + \bar{P}_{n});
Om12_n := -\tau2^{-1} Q_{n};
Om21_n := (\tau1^{2} - \tau2^{2}) \tau2^{-1} Q_{n} + \tau1 (P_{n} + \bar{P}_{n});
Om22_n :=  \tau1 \tau2^{-1} Q_{n} + (1/2)(P_{n} + \bar{P}_{n});

# Verify: Om11 + Om22 = 0 (tracelessness from det M = 1)
trace_check := @(Om11_m) + @(Om22_m);
collect_terms(_);
print("Tr(Omega) =", str(trace_check), " (should vanish)")

In [ ]:
# hatR_{mn12} = M_{kl}(Om_m^k_2 Om_n^l_1 - Om_m^k_1 Om_n^l_2)
# Cadabra builds the full sum over k,l with M_{kl} coefficients.

hatR_mn12 :=
  (\tau1^{2} + \tau2^{2}) \tau2^{-1} ( @(Om12_m) @(Om11_n) - @(Om11_m) @(Om12_n) )
+ \tau1 \tau2^{-1} ( @(Om12_m) @(Om21_n) - @(Om11_m) @(Om22_n)
                   + @(Om22_m) @(Om11_n) - @(Om21_m) @(Om12_n) )
+ \tau2^{-1} ( @(Om22_m) @(Om21_n) - @(Om21_m) @(Om22_n) );

distribute(_);
collect_terms(_);
print("hatR_{mn12} =", str(hatR_mn12))

In [ ]:
# Express in P, Pbar by substituting Q = (i/2)(P - Pbar)
subs_Q := { Q_{m} -> (1/2) \imath (P_{m} - \bar{P}_{m}),
            Q_{n} -> (1/2) \imath (P_{n} - \bar{P}_{n}) };

hatR_mn12_PQ := @(hatR_mn12);
substitute(_, subs_Q);
distribute(_);
collect_terms(_);
print("hatR_{mn12} in P,Pbar =", str(hatR_mn12_PQ))

## Step 4: $\hat{R}_{1212}$ (pure fiber block)

$$\hat{R}_{ijkl} = g_{mn}\bigl(\hat{\Gamma}^m_{il}\hat{\Gamma}^n_{jk} - \hat{\Gamma}^m_{ik}\hat{\Gamma}^n_{jl}\bigr)$$

with $\hat{\Gamma}^m_{ij} = -\tfrac{1}{2}g^{mn}\partial_n\mathcal{M}_{ij}$
computed directly from $\partial_n\mathcal{M}_{ij}$.

In [ ]:
# hatGamma^m_{ij} = -(1/2) g^{mn} d_n M_{ij}

HG22 := -(1/2) g^{m n} @(dM22_n);
HG12 := -(1/2) g^{m n} @(dM12_n);
HG11 := -(1/2) g^{m n} @(dM11_n);

print("HG^m_22 =", str(HG22))
print("HG^m_12 =", str(HG12))
print("HG^m_11 =", str(HG11))

In [ ]:
# hatR_{1212} = g_{mp}(HG^m_{12} HG^p_{12} - HG^m_{11} HG^p_{22})
# Second copy of HG with independent dummy indices (p,q).

dM22_q := \tau2^{-1} (P_{q} + \bar{P}_{q});
dM12_q := -2 Q_{q} + \tau1 \tau2^{-1} (P_{q} + \bar{P}_{q});
dM11_q := -4 \tau1 Q_{q} + (\tau1^{2} - \tau2^{2}) \tau2^{-1} (P_{q} + \bar{P}_{q});

HG22b := -(1/2) g^{p q} @(dM22_q);
HG12b := -(1/2) g^{p q} @(dM12_q);
HG11b := -(1/2) g^{p q} @(dM11_q);

hatR1212 := g_{m p} ( @(HG12) @(HG12b) - @(HG11) @(HG22b) );
distribute(_);
collect_terms(_);
print("hatR_{1212} =", str(hatR1212))

In [ ]:
# Express in P, Pbar
subs_Qall := { Q_{m} -> (1/2) \imath (P_{m} - \bar{P}_{m}),
               Q_{n} -> (1/2) \imath (P_{n} - \bar{P}_{n}),
               Q_{p} -> (1/2) \imath (P_{p} - \bar{P}_{p}),
               Q_{q} -> (1/2) \imath (P_{q} - \bar{P}_{q}) };

hatR1212_PQ := @(hatR1212);
substitute(_, subs_Qall);
distribute(_);
collect_terms(_);
print("hatR_{1212} in P,Pbar =", str(hatR1212_PQ))

## Step 5: Complex frame rotation

Convert from real torus coordinates $(x^1, x^2)$ to the complex frame
$z = w/\sqrt{\tau_2}$ where $w = \tau x^1 + x^2$ (matching our $\mathcal{M}$ convention).
This gives $\mathcal{M}_{z\bar{z}} = 1/2$.

The combined vielbein $f^i_z$ (coordinate $\to$ flat complex) has
$\det(f) = -i/2$.

**Pure fiber**: $\hat{R}_{z\bar{z}z\bar{z}} = |\det f|^2 \cdot \hat{R}_{1212} = \tfrac{1}{4}\hat{R}_{1212}$

**Spacetime-fiber**: $\hat{R}_{mn,z\bar{z}} = \det(f) \cdot \hat{R}_{mn12} = -\tfrac{i}{2}\hat{R}_{mn12}$

**Mixed**: $R_{m,z,n,\bar{z}} = f^i_z\,f^j_{\bar{z}}\,R_{imjn}$
with $4\tau_2\,R_{m,z,n,\bar{z}} = R_{1m1n} - 2\tau_1\,R_{1m2n} + |\tau|^2\,R_{2m2n}$.
After expanding, all $D_mP_n$ terms cancel and the result is
$R_{m,z,n,\bar{z}} = -\tfrac{1}{2}(P_m\bar{P}_n + \bar{P}_m P_n)$.

In [ ]:
print("================================================")
print("12d Riemann tensor - Results")
print("================================================")
print("")
print("Convention: z,zbar flat complex frame, M_{z zbar} = 1/2")
print("")
print("(i)   hatR_{z zbar z zbar}  = -(1/4) hatR_{1212}")
print("      cadabra: hatR_{1212} =", str(hatR1212))
print("")
print("(ii)  hatR_{mn z zbar}      = -(i/2) hatR_{mn12}")
print("      cadabra: hatR_{mn12} =", str(hatR_mn12))
print("")
print("(iii) R_{m z n zbar}        = -(1/2)(P_m Pbar_n + Pbar_m P_n)")
print("      (from linear combination of R_{imjn} above)")
print("")
print("(iv)  R_{m z n z}           = complex, involves DP_{mn}")
print("      4 tau2 R_{mznz} = -R_{1m1n} + 2 tbar R_{1m2n} - tbar^2 R_{2m2n}")